# Tutorial 1: **Notebook** - *INV*

**By 
gLayout Team**

__Content creators:__ Subham Pal, Saptarshi Ghosh

__Content reviewers:__ Mehedi Saligne

___
# Tutorial Objectives

This notebook is a tutorial on-

- **Importing** and **Placement** of FETs and other macros/Pcells with relative coordinates + Placing (and connecting) Via_stack on the ports of the FETs + encircling them with padrings.
- **Routing** between the placed *Vias* with *C_*, *L_*, *Straight_* and *smart* Routes and explaining the differences between these strategies. Particularly explaning the East/West/North/South angles, (For Example, why C has to parallel but L has to be perpendicular sides) and *Placing and connecting PINs for future LVS runs

## **Target** **Block** : **Inverter Cell**

The CMOS inverter is a fundamental digital circuit building block, widely used due to its low static power consumption, high noise immunity, and fast switching characteristics. It consists of a complementary pair of MOSFETs: one PMOS and one NMOS transistor, with their gates and drains connected together. The source of the PMOS is connected to the supply voltage (VDD), while the source of the NMOS is connected to ground (VSS).

### **Operation Principle**
- **Input Low (Logic 0):** The PMOS transistor turns ON and the NMOS turns OFF. This connects the output (VOUT) to VDD, producing a logic high (1) at the output.
- **Input High (Logic 1):** The NMOS transistor turns ON and the PMOS turns OFF. This connects the output to ground, producing a logic low (0) at the output.

This complementary switching action ensures that the output is always the logical inverse of the input, making the CMOS inverter a true logic inverter.

![](_images/CMOS_Inverter.png)

```bibtex
Wikimedia Commons contributors. (2010). CMOS Inverter Schematic [Diagram]. Wikimedia Commons. https://commons.wikimedia.org/wiki/File:CMOS_Inverter.svg
```


## **Layout generation**
let's go through the step by step procedure to generate LVS and DRC clean layout of a INV cell.


### *Creating environment for live GDSII viewer*
You can see the generated GDSII file immediately inside this notebook. You can also view it in Klayout for better understanding. Code for viewing it in Klayout is added but commented out.

In [ ]:
## Activate if necessary to install following packages

# !pip install svgutils
# !pip install ipywidgets

This cell only required for [IIC-OSIC Docker](https://github.com/iic-jku/IIC-OSIC-TOOLS). It imports all neceaasry environment varriable to current session

In [ ]:
import os
import subprocess

# Run a shell, source .bashrc, then printenv
cmd = 'bash -c "source ~/.bashrc && printenv"'
result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
env_vars = {}
for line in result.stdout.splitlines():
    if '=' in line:
        key, value = line.split('=', 1)
        env_vars[key] = value

# Now, update os.environ with these
os.environ.update(env_vars)

In [ ]:
import gdstk
import svgutils.transform as sg
import IPython.display
from IPython.display import clear_output
import ipywidgets as widgets

# Redirect all outputs here
hide = widgets.Output()

def display_gds(gds_file, path,scale = 3):
  
  # Generate an SVG image
  top_level_cell = gdstk.read_gds(gds_file).top_level()[0]
  top_level_cell.write_svg(os.path.join(path,'out.svg'))
    
  # Scale the image for displaying
  fig = sg.fromfile(os.path.join(path,'out.svg'))
  fig.set_size((str(float(fig.width) * scale), str(float(fig.height) * scale)))
  fig.save(os.path.join(path,'out.svg'))

  # Display the image
  IPython.display.display(IPython.display.SVG(os.path.join(path,'out.svg')))
  os.remove(os.path.join(path,'out.gds'))

def display_component(component,path,scale = 3):
  # Save to a GDS file
  with hide:
    component.write_gds(os.path.join(path,'out.gds'))
  display_gds(os.path.join(path,'out.gds'),path,scale)

In [ ]:
from glayout import MappedPDK, sky130 , gf180
#from gdsfactory.cell import cell
from gdsfactory import Component
from gdsfactory.components import text_freetype, rectangle

In [ ]:
from glayout import nmos, pmos
from glayout import via_stack
from glayout import rename_ports_by_orientation
from glayout import tapring

In [ ]:
from glayout.util.comp_utils import evaluate_bbox, prec_center, prec_ref_center, align_comp_to_port
from glayout.util.port_utils import add_ports_perimeter,print_ports
from glayout.util.snap_to_grid import component_snap_to_grid
from glayout.spice.netlist import Netlist

In [ ]:
from glayout.routing.straight_route import straight_route
from glayout.routing.c_route import c_route
from glayout.routing.L_route import L_route

INV has two complimentary fets as shown in the schematic. One is PMOS and another is NMOS. Their operations in INV are explained above. Lets define arguments for the FETs

In [ ]:
nmos_kwargs = {
    "with_tie": True,
    "with_dnwell": True,
    "sd_route_topmet": "met2",
    "gate_route_topmet": "met2",
    "sd_route_left": True,
    "rmult": None,
    "gate_rmult": 1,
    "interfinger_rmult": 1,
    "substrate_tap_layers": ("met2","met1"),
    "dummy_routes": True
}

pmos_kwargs = {
    "with_tie": True,
    "dnwell": False,
    "sd_route_topmet": "met2",
    "gate_route_topmet": "met2",
    "sd_route_left": True,
    "rmult": None,
    "gate_rmult": 1,
    "interfinger_rmult": 1,
    "substrate_tap_layers": ("met2","met1"),
    "dummy_routes": True
}

There is no need to manually define the kwargs for your code. This is only done for this notebook implementation.


Now, Let's create a list for parameters which we need for our function-

* pdk: pdk to use
* placement: either "horizontal" or "vertical"
* width: (input fet, feedback fet)
* length: (input fet, feedback fet)
* fingers: (input fet, feedback fet)
* multipliers: (input fet, feedback fet)
* dummy_1: dummy for input fet
* dummy_2: dummy for feedback fet


In [ ]:
inverter_config={
        "pdk": gf180, # pdk to use
        "placement" : "horizontal", # the two fets can be placed either vertically or horizontally
        "width": (3,3), # width of the input fet and feedback fet respectively.
        "length": (None,None), # length of the input fet and feedback fet respectively. None refers to the min length in the pdk.
        "fingers": (1,1), # no. of fingers of the input fet and feedback fet respectively.
        "multipliers": (1,1), #no. of multipliers of the input fet and feedback fet respectively.
        "dummy_1": (True,True), # dummy pattern for input fet (left,right)
        "dummy_2": (True,True), # dummy pattern for the feedback fet (left,right)
        "tie_layers1": ("met2","met1"), #tapring metal layers 
        "tie_layers2": ("met2","met1"), #tapring metal layers 
        "sd_rmult":1, # thickness of the sd metal layer.
}

### First: We create a top level component and add a name

In [ ]:
top_level = Component(name="inverter")

### Two fets

For our INV block, we do not want deep nwell. However nmos and pmos primitives have different parameter name for deepnwell.
For pmos it is set to false by default.We need to set it to false if the device type is given as nmos.
We could've just set it properly in our nmos_kwargs,
but since you will probably not code the layout for a notebook, we show a general way of solving this issue.


In [ ]:
pdk=inverter_config["pdk"]
pdk.activate()

In [ ]:
width=inverter_config["width"]
length=inverter_config["length"]
fingers=inverter_config["fingers"]
multipliers=inverter_config["multipliers"]

dummy_1=inverter_config["dummy_1"]
dummy_2=inverter_config["dummy_2"]
tie_layers1=inverter_config["tie_layers1"]
tie_layers2=inverter_config["tie_layers2"]
sd_rmult=inverter_config["sd_rmult"]

In [ ]:
#fet_1 is the input fet
fet_P = pmos(pdk, width=width[0], fingers=fingers[0], multipliers=multipliers[0], with_dummy=dummy_1, with_substrate_tap=False, length=length[0], tie_layers=tie_layers1, sd_rmult=sd_rmult, **pmos_kwargs )
#fet_2 is the feedback fet
nmos_kwargs["with_dnwell"] = False
fet_N = nmos(pdk, width=width[1], fingers=fingers[1], multipliers=multipliers[1], with_dummy=dummy_2, with_substrate_tap=False, length=length[1], tie_layers=tie_layers2, sd_rmult=sd_rmult, **nmos_kwargs)

Adding the references of the FETs to the top level component

In [ ]:
fet_P_ref = top_level << fet_P
fet_N_ref = top_level << fet_N

fet_P_ref.name = "fet_P"
fet_N_ref.name = "fet_N"

#let's take a look at our current state of layout
## To see in Klayout via Klive
#top_level.show()
display_component(top_level, scale = 5,path="../../")
## If you want to save the gds file at this stage
#primary_gds = top_level.write_gds("before_placement.gds")

### Placement

When a cell is referenced to the top level, the center of the cell is aligned to the (0,0) co-ordinate.
As of now, both of our fets are centered at (0,0). We need place them properly depending on the placement parameter.

In [ ]:
placement=inverter_config["placement"]
ref_dimensions = evaluate_bbox(fet_N)


evaluate_bbox returns maximum width and maximum height of the block as a tuple(width,height).

So here we have `ref_dimensions=(x-dimension of fet_2, y-dimension of fet_2)`

For horizontal placement , we need to move the fet_2 in rightward direction.
We use the movex function, which moves the center of the block in x-directon upto a certain distance. We need to find out this distance.
Remember that both fet_1 and fet_2 are centered at (0,0) co-ordinate right now.
At first we move the center of fet_2 to rightmost point of fet_1 using the xmax function, i.e, (fet_1_ref.xmax,0).
Now we will find that left half of fet_2 is overlapping with fet_1, so we need to move it at rightward direction again.We move it by ref_dimensionsp[0]/2 distance.
Now we see that the rightmost point of fet_1 and leftmost point of fet_2 have no seperation between them.
This will cause a drc error. So we move it again by max seperation needed between two metal layers for drc free layout.
We move it by 1um again for leaving some routing space for future use.

Vertical placement can be done in similar way by using movey function.

In [ ]:
if placement == "horizontal":
    fet_N_ref.movex(fet_P_ref.xmax)
    display_component(top_level, scale = 2,path="../../")
    fet_N_ref.movex(ref_dimensions[0]/2)
    display_component(top_level, scale = 2,path="../../")
    fet_N_ref.movex(pdk.util_max_metal_seperation()+1)
elif placement == "vertical":
    display_component(top_level, scale = 2,path="../../")
    fet_N_ref.movey(fet_P_ref.ymin - ref_dimensions[1]/2 - pdk.util_max_metal_seperation()-1)
else:
        raise ValueError("Placement must be either 'horizontal' or 'vertical'.")
    
# let's see how our block looks after placement
display_component(top_level, scale = 2,path="../../")
## To see in Klayout via Klive
#top_level.show()

 ### Routing

Now that our fets are properly placed, we need to make proper connections between ports (i.e. Gates, Sources and Drains of the FETs).
Each port has four directions- East,North,West,South which means 0,90,180,270 degrees respectively.
You can connect directly to a fet port. For better felxibility in routing, we choose add some intermediate vias here.

![](_images/port_route.png)


In [ ]:
viam2m3 = via_stack(pdk, "met2", "met3", centered=True) #met2 is the bottom layer. met3 is the top layer.

#we need four such vias
drain_P_via = top_level << viam2m3
source_P_via = top_level << viam2m3
gate_P_via = top_level << viam2m3

drain_N_via = top_level << viam2m3
gate_N_via = top_level << viam2m3

The move function moves the center of a specific cell to a specific co-ordinate.
The co-ordinates of the center of a port can be accesed by using cell_name.ports["port_name"].center
Using these two and other techniques previously discussed, we properly place these 4 vias.

In [ ]:
drain_P_via.move(fet_P_ref.ports["multiplier_0_drain_W"].center).movex(-1.5)
drain_N_via.move(fet_N_ref.ports["multiplier_0_drain_W"].center).movex(-1.5)

source_P_via.move(fet_P_ref.ports["multiplier_0_source_E"].center).movex(1.5)

gate_P_via.move(fet_P_ref.ports["multiplier_0_gate_E"].center).movex(1)
gate_N_via.move(fet_N_ref.ports["multiplier_0_gate_E"].center).movex(1)

# let's see where those vias are now
## To see in Klayout via Klive
#top_level.show()
display_component(top_level, scale = 3,path="../../")

Now that the vias are placed, time to make metal routing.
There are three basic types of routing available:

- `straight_route` - the ports must be straight to each other, i.e, opposite direction and same x or y coordinate.
- `L_route` - has a 90&deg; bend, ports must be perpendicular to each other.
- `c_route` -  has two 90&deg; bends(c shaped), ports must have same direction.

Each routing function has multiple parameters. See [here](https://github.com/idea-fasoc/OpenFASOC/tree/main/openfasoc/generators/glayout/glayout/flow/routing)

![](_images/route_direction.png)

Bonus: You can see all avaliable ports by `top_level.pprint_ports()`

From the schematic, we see that we need to connect these two nodes-

1. **Connect the Source of M1 (PMOS) to VDD:**  
   Insert an intermediate via near the source of M1 and connect it to the VDD rail using a metal trace.

2. **Connect the Source of M2 (NMOS) to GND:**  
   Insert an intermediate via near the source of M2 and connect it to the GND rail using a metal trace.

3. **Connect the Drains of M1 and M2 Together (Output Node):**  
   Insert intermediate vias near the drain of M1 (PMOS) and the drain of M2 (NMOS), then connect them together with a metal trace. This node forms the inverter output.

4. **Connect the Gates of M1 and M2 Together (Input Node):**  
   Insert intermediate vias near the gate of M1 (PMOS) and the gate of M2 (NMOS), then connect them together with a metal trace. This node forms the inverter input.

In [ ]:
top_level << straight_route(pdk, fet_P_ref.ports["multiplier_0_gate_E"], gate_P_via.ports["bottom_met_N"])
top_level << straight_route(pdk, fet_N_ref.ports["multiplier_0_gate_E"], gate_N_via.ports["bottom_met_W"])
top_level << straight_route(pdk, fet_P_ref.ports["multiplier_0_source_E"], source_P_via.ports["bottom_met_W"])
top_level << straight_route(pdk, fet_P_ref.ports["multiplier_0_drain_W"], drain_P_via.ports["bottom_met_E"])
top_level << straight_route(pdk, fet_N_ref.ports["multiplier_0_drain_W"], drain_N_via.ports["bottom_met_E"])

if placement == "horizontal":
    top_level << c_route(pdk, gate_P_via.ports["top_met_S"], gate_N_via.ports["top_met_S"],extension=1.2*max(width[0],width[1]), width1=0.32, width2=0.32, cwidth=0.32, e1glayer="met3", e2glayer="met3", cglayer="met2")
    top_level << c_route(pdk, drain_P_via.ports["top_met_N"], drain_N_via.ports["top_met_N"],extension=1.2*max(width[0],width[1]), width1=0.32, width2=0.32, cwidth=0.32, e1glayer="met3", e2glayer="met3", cglayer="met2")
elif placement == "vertical":
    top_level << straight_route(pdk, gate_P_via.ports["top_met_S"], gate_N_via.ports["top_met_S"])
    top_level << straight_route(pdk, drain_P_via.ports["top_met_N"], drain_N_via.ports["top_met_N"])
else:
        raise ValueError("Placement must be either 'horizontal' or 'vertical'.")
    
try:
    top_level << straight_route(pdk, fet_N_ref.ports["multiplier_0_source_W"], fet_N_ref.ports["tie_W_top_met_W"], glayer1=tie_layers2[1], fullbottom=True)
except:
    pass

# let's see how those vias are now routed
#top_level.show()
display_component(top_level, scale = 3,path="../../")

Notice how in the above lines, ports used in straight routing has the opposite direction, e.g. (E,W) but ports used in c_route has the same direction (N,N).

### Renaming Ports

We need to rename ports and add them to the top level component for ease of routing in hierarchial design.
Here we rename ports by adding appropriate prefix to the all ready existing port names.


In [ ]:
top_level.add_ports(fet_P_ref.get_ports_list(), prefix="P_")
top_level.add_ports(fet_N_ref.get_ports_list(), prefix="N_")
top_level.add_ports(drain_P_via.get_ports_list(), prefix="P_drain_")
top_level.add_ports(source_P_via.get_ports_list(), prefix="P_source_")
top_level.add_ports(gate_P_via.get_ports_list(), prefix="P_gate_")
top_level.add_ports(drain_N_via.get_ports_list(), prefix="N_drain_")
top_level.add_ports(gate_N_via.get_ports_list(), prefix="N_gate_")

In [ ]:
component = component_snap_to_grid(rename_ports_by_orientation(top_level))
    
# Now that our block is complete, let's see how it looks now!!
#component.show()
display_component(component, scale = 2,path="../../")
#component.show()
##inv_gds = component.write_gds("inv.gds")

Printing List of Ports of the Component

In [ ]:
## To see all Ports
# component.pprint_ports()

## To see selected ports
# for s in component.get_ports_list():
#     if len(s.name.split("_")) <4:
#         if set(["P","gate"]).issubset(set(s.name.split("_"))):
#             print(s.name, s.port_type , s.orientation, s.center, s.width, s.layer)

### Add Pins
We need to add i/o pins, power supply pins and bias pins to a circuit. This will be needed later to do LVS check(Layout Vs Schematic) and parasitic extraction(PEX).
Our inverter has four pins (Nodes) - VIN, VOUT, VDD, VBULK

In [ ]:
psize=(0.5,0.5)
# list that will contain all port/comp info
move_info = list()
# create labels and append to info list

In [ ]:
# gnd
gndlabel = rectangle(layer=pdk.get_glayer("met3_pin"),size=psize,centered=True).copy()
gndlabel.add_label(text="VBULK",layer=pdk.get_glayer("met3_label"))
move_info.append((gndlabel,component.ports["N_tie_N_top_met_N"],None))
#gnd_ref = top_level << gndlabel;



#suply
suplabel = rectangle(layer=pdk.get_glayer("met3_pin"),size=psize,centered=True).copy()
suplabel.add_label(text="VDD",layer=pdk.get_glayer("met3_pin"))
move_info.append((suplabel,component.ports["P_source_top_met_N"],None))
#sup_ref = top_level << suplabel;


# output
outputlabel = rectangle(layer=pdk.get_glayer("met3_pin"),size=psize,centered=True).copy()
outputlabel.add_label(text="VOUT",layer=pdk.get_glayer("met3_pin"))
move_info.append((outputlabel,component.ports["P_drain_top_met_N"],None))
#op_ref = top_level << outputlabel;


# input
inputlabel = rectangle(layer=pdk.get_glayer("met3_pin"),size=psize,centered=True).copy()
inputlabel.add_label(text="VIN",layer=pdk.get_glayer("met3_pin"))
move_info.append((inputlabel,component.ports["P_gate_top_met_N"], None))
#ip_ref = top_level << inputlabel;


for comp, prt, alignment in move_info:
        alignment = ('c','b') if alignment is None else alignment
        compref = align_comp_to_port(comp, prt, alignment=alignment)
        top_level.add(compref)
component = top_level.flatten()


component.show()
display_component(top_level, scale =5,path="../../")

## Note
Now, lets construct the Pcell and export this code as a `.py` file. This is how your cell contributions should be structured. 

In [ ]:
inv_code_string = """
from glayout import MappedPDK, sky130 , gf180
from gdsfactory.cell import cell
from gdsfactory import Component
from gdsfactory.components import text_freetype, rectangle

from glayout import nmos, pmos
from glayout import via_stack
from glayout import rename_ports_by_orientation
from glayout import tapring

from glayout.util.comp_utils import evaluate_bbox, prec_center, prec_ref_center, align_comp_to_port
from glayout.util.port_utils import add_ports_perimeter,print_ports
from glayout.util.snap_to_grid import component_snap_to_grid
from glayout.spice.netlist import Netlist

from glayout.routing.straight_route import straight_route
from glayout.routing.c_route import c_route
from glayout.routing.L_route import L_route

import os
import subprocess

# Run a shell, source .bashrc, then printenv
cmd = 'bash -c "source ~/.bashrc && printenv"'
result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
env_vars = {}
for line in result.stdout.splitlines():
    if '=' in line:
        key, value = line.split('=', 1)
        env_vars[key] = value

# Now, update os.environ with these
os.environ.update(env_vars)

def add_inv_labels(
    inv_in: Component,
    pdk: MappedPDK,
    ) -> Component:
    
    inv_in.unlock()

    psize=(0.5,0.5)
    # list that will contain all port/comp info
    move_info = list()
    # create labels and append to info list

    # gnd
    gndlabel = rectangle(layer=pdk.get_glayer("met3_pin"),size=psize,centered=True).copy()
    gndlabel.add_label(text="VBULK",layer=pdk.get_glayer("met3_label"))
    move_info.append((gndlabel,inv_in.ports["N_tie_N_top_met_N"],None))
    #gnd_ref = top_level << gndlabel;



    #suply
    suplabel = rectangle(layer=pdk.get_glayer("met3_pin"),size=psize,centered=True).copy()
    suplabel.add_label(text="VDD",layer=pdk.get_glayer("met3_pin"))
    move_info.append((suplabel,inv_in.ports["P_source_top_met_N"],None))
    #sup_ref = top_level << suplabel;


    # output
    outputlabel = rectangle(layer=pdk.get_glayer("met3_pin"),size=psize,centered=True).copy()
    outputlabel.add_label(text="VOUT",layer=pdk.get_glayer("met3_pin"))
    move_info.append((outputlabel,inv_in.ports["P_drain_top_met_N"],None))
    #op_ref = top_level << outputlabel;


    # input
    inputlabel = rectangle(layer=pdk.get_glayer("met3_pin"),size=psize,centered=True).copy()
    inputlabel.add_label(text="VIN",layer=pdk.get_glayer("met3_pin"))
    move_info.append((inputlabel,inv_in.ports["P_gate_top_met_N"], None))
    #ip_ref = top_level << inputlabel;


    for comp, prt, alignment in move_info:
            alignment = ('c','b') if alignment is None else alignment
            compref = align_comp_to_port(comp, prt, alignment=alignment)
            inv_in.add(compref)
    
    return inv_in.flatten()
@cell
def inverter(
        pdk: MappedPDK,
        placement: str = "horizontal",
        width: tuple[float,float] = (3,3),
        length: tuple[float,float] = (None,None),
        fingers: tuple[int,int] = (1,1),
        multipliers: tuple[int,int] = (1,1),
        dummy_1: tuple[bool,bool] = (True,True),
        dummy_2: tuple[bool,bool] = (True,True),
        tie_layers1: tuple[str,str] = ("met2","met1"),
        tie_layers2: tuple[str,str] = ("met2","met1"),
        sd_rmult: int=1,
        **kwargs
        ) -> Component:

    pdk.activate()
    
    #top level component
    top_level = Component(name="inverter")

    #two fets
    fet_P = pmos(pdk, width=width[0], fingers=fingers[0], multipliers=multipliers[0], with_dummy=dummy_1, with_substrate_tap=False, length=length[0], tie_layers=tie_layers1, sd_rmult=sd_rmult, **kwargs )
    fet_N = nmos(pdk, width=width[1], fingers=fingers[1], multipliers=multipliers[1], with_dummy=dummy_2, with_substrate_tap=False, length=length[1], tie_layers=tie_layers2, sd_rmult=sd_rmult, with_dnwell=False, **kwargs)
    
    fet_P_ref = top_level << fet_P
    fet_N_ref = top_level << fet_N 
    fet_P_ref.name = "fet_P"
    fet_N_ref.name = "fet_N"

    #Relative move
    ref_dimensions = evaluate_bbox(fet_N)
    if placement == "horizontal":
        fet_N_ref.movex(fet_P_ref.xmax + (ref_dimensions[0]/2) + pdk.util_max_metal_seperation()+1)
    elif placement == "vertical":
        fet_N_ref.movey(fet_P_ref.ymin - ref_dimensions[1]/2 - pdk.util_max_metal_seperation()-1)
    else:
        raise ValueError("Placement must be either 'horizontal' or 'vertical'.")

    #Routing
    viam2m3 = via_stack(pdk, "met2", "met3", centered=True) #met2 is the bottom layer. met3 is the top layer.
    
    #we need four such vias
    drain_P_via = top_level << viam2m3
    source_P_via = top_level << viam2m3
    gate_P_via = top_level << viam2m3

    drain_N_via = top_level << viam2m3
    gate_N_via = top_level << viam2m3
    
        
    drain_P_via.move(fet_P_ref.ports["multiplier_0_drain_W"].center).movex(-1.5)
    drain_N_via.move(fet_N_ref.ports["multiplier_0_drain_W"].center).movex(-1.5)

    source_P_via.move(fet_P_ref.ports["multiplier_0_source_E"].center).movex(1.5)

    gate_P_via.move(fet_P_ref.ports["multiplier_0_gate_E"].center).movex(1)
    gate_N_via.move(fet_N_ref.ports["multiplier_0_gate_E"].center).movex(1)


    top_level << straight_route(pdk, fet_P_ref.ports["multiplier_0_gate_E"], gate_P_via.ports["bottom_met_N"])
    top_level << straight_route(pdk, fet_N_ref.ports["multiplier_0_gate_E"], gate_N_via.ports["bottom_met_W"])
    top_level << straight_route(pdk, fet_P_ref.ports["multiplier_0_source_E"], source_P_via.ports["bottom_met_W"])
    top_level << straight_route(pdk, fet_P_ref.ports["multiplier_0_drain_W"], drain_P_via.ports["bottom_met_E"])
    top_level << straight_route(pdk, fet_N_ref.ports["multiplier_0_drain_W"], drain_N_via.ports["bottom_met_E"])

    if placement == "horizontal":
        top_level << c_route(pdk, gate_P_via.ports["top_met_S"], gate_N_via.ports["top_met_S"],extension=1.2*max(width[0],width[1]), width1=0.32, width2=0.32, cwidth=0.32, e1glayer="met3", e2glayer="met3", cglayer="met2")
        top_level << c_route(pdk, drain_P_via.ports["top_met_N"], drain_N_via.ports["top_met_N"],extension=1.2*max(width[0],width[1]), width1=0.32, width2=0.32, cwidth=0.32, e1glayer="met3", e2glayer="met3", cglayer="met2")
    elif placement == "vertical":
        top_level << straight_route(pdk, gate_P_via.ports["top_met_S"], gate_N_via.ports["top_met_S"])
        top_level << straight_route(pdk, drain_P_via.ports["top_met_N"], drain_N_via.ports["top_met_N"])
    else:
        raise ValueError("Placement must be either 'horizontal' or 'vertical'.")

    try:
        top_level << straight_route(pdk, fet_N_ref.ports["multiplier_0_source_W"], fet_N_ref.ports["tie_W_top_met_W"], glayer1=tie_layers2[1], fullbottom=True)
    except:
        pass

    top_level.add_ports(fet_P_ref.get_ports_list(), prefix="P_")
    top_level.add_ports(fet_N_ref.get_ports_list(), prefix="N_")
    top_level.add_ports(drain_P_via.get_ports_list(), prefix="P_drain_")
    top_level.add_ports(source_P_via.get_ports_list(), prefix="P_source_")
    top_level.add_ports(gate_P_via.get_ports_list(), prefix="P_gate_")
    top_level.add_ports(drain_N_via.get_ports_list(), prefix="N_drain_")
    top_level.add_ports(gate_N_via.get_ports_list(), prefix="N_gate_")

    return component_snap_to_grid(rename_ports_by_orientation(top_level))

if __name__ == "__main__":
\tcomp = inverter(gf180)\n
\t# comp.pprint_ports()\n
\tcomp = add_inv_labels(comp, gf180)\n
\tcomp.name = "INV"\n
\tcomp.write_gds('out_INV.gds')\n
\tcomp.show()\n
\tprint("...Running DRC...")\n
\tdrc_result = gf180.drc_magic(comp, "INV")\n
\tdrc_result = gf180.drc(comp)\n
"""

inv_init_string = """
###Glayout FVF Cell.


from .my_INV import inverter,add_inv_labels

__all__ = [
    'inverter',
    'add_inv_labels',
] 
"""

directory = "../../INV/"
os.makedirs(directory, exist_ok=True)

# Save to a .py file
with open(directory + "my_INV.py", "w") as file:
    file.write(inv_code_string)

with open(directory + "__init__.py", "w") as file:
    file.write(inv_init_string)

### Run DRC with Magic
Design Rule Check ensures that the physical layout of an integrated circuit adheres to the manufacturing constraints defined by the foundry, such as minimum spacing, width, and enclosure rules. `Magic` is the tool we use for DRC here.

In [ ]:
component.name="inv"
drc_result = gf180.drc_magic(component, component.name)

### DRC Checking Using External Tools (KLayout)
Design Rule Check (DRC) is the process of ensuring that a particular layout does not violate the constraints or _design rules_ imposed by the PDK.

[Klayout](https://klayout.de/) is a GDSII viewer and editor that also has a DRC feature. The design rules for the PDK, in this case the GF180 PDK, are described in a `.lydrc` file.

The following cell runs DRC on the component generated in the previous cell. The number of DRC errors reported will be displayed at the end of the output.

In [ ]:
drc_result = gf180.drc(component)